# 06 · Full baseline run

The full baseline answers every item of `run_benchmarks` x `run_splits` at each loop count in `R_grid`, with the same code path as `scripts/mvp.py full`. Items are sharded (`items[i::n]`), one shard per GPU group, and each shard writes its own resumable JSONL. This notebook first checks batched generation against batch 1, then runs the full run only when `RUN_FULL` is True, then summarises the result files. The test split stays off until stage S12.

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os
import socket
import statistics
from dataclasses import asdict

In [ ]:
from adaloop import api  # importing api first; setup_notebook puts the repo root on sys.path

GPU_GROUP = None  # None = first free pair in gpu_groups; 0 or 1 forces a pair; changing it needs a kernel restart
ROOT, cfg = api.setup_notebook("configs/full_baseline.yaml", gpu_group=GPU_GROUP)
print("ROOT", ROOT, "| config_hash", api.config_hash(cfg), "| git", api.git_hash())

In [ ]:
# torch is imported here because the GPU list needs it; the other third-party imports come after this cell
import torch

IS_INARA = "inara" in socket.gethostname()
print("host", socket.gethostname(), "| on Inara:", IS_INARA)
print("chosen GPUs, physical ids CUDA_VISIBLE_DEVICES =", os.environ.get("CUDA_VISIBLE_DEVICES"))
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        total_gib = torch.cuda.get_device_properties(i).total_memory / 1024**3
        print(f"chosen GPU {i} (torch index) | {torch.cuda.get_device_name(i)} | {total_gib:.0f} GiB")
else:
    print("no CUDA GPU visible: the model steps will fail, the rest of the notebook can still be read")
print("gpu_groups", cfg["gpu_groups"], "| only these physical GPUs (0-3 on Inara) may be used")

In [ ]:
import pandas as pd

In [ ]:
# Settings: the full run is off by default because it is long
N_CHECK = 8  # dev items for the batch check: half GQA, half TextVQA
R_CHECK = 3  # loop passes for the batch check
RUN_FULL = False  # step 2: True runs shard SHARD in this kernel
SHARD = (0, 2)  # step 2: (i, n) run by this kernel; the terminal commands in step 2 do the rest
N_COST = 16  # step 4: dev items for timing, half GQA, half TextVQA
R_RATIO = (1, 5)  # step 4: prefill time ratio is R_RATIO[1] / R_RATIO[0]
N_REPEATS = 3  # step 4: timed prefill calls per item and R, after one warmup
RATIO_TOL = 0.25  # step 4: allowed gap between measured and analytic ratio (DEC3 §6)
R_THROUGHPUT = 3  # step 4: loop passes for the throughput comparison
THROUGHPUT_BATCHES = (1, cfg["batch_size"])  # step 4

### Step-1:- Batch check: batch 1 vs batched generation

Batched generation pads the prompts to one length. Padding and batched GPU kernels can change the numerics, and a near-tied
choice can flip, so `batch_size` may give a different answer than batch 1. This step runs the same dev items at batch 1 and at
`batch_size` from the config, then compares the predictions. A mismatch must be reported, not ignored: the table shows which
items differ, and the full run should not start on batched rows until that is decided.

In [ ]:
print("[STAGE] load model")
vlm = api.load_model(cfg)

In [ ]:
print("[STAGE] batch check inputs")
# half GQA, half TextVQA, so one batch mixes both benchmarks
check_items = (
    api.load_gqa(cfg, "dev", limit=N_CHECK // 2, root=ROOT)
    + api.load_textvqa(cfg, "dev", limit=N_CHECK // 2, root=ROOT)
)
assert len(check_items) == N_CHECK, len(check_items)
print(f"{len(check_items)} items | batch sizes to compare: 1 and {cfg['batch_size']}")

In [ ]:
BATCH = cfg["batch_size"]
# one file per batch size; the name ends with the hash of the same run config the full run uses
check_rows = {}
for batch_size in (1, BATCH):
    run_cfg = api.full_run_config(cfg, batch_size)
    out = ROOT / cfg["out_dir"] / f"check_dev_n{N_CHECK}_R{R_CHECK}_b{batch_size}_{api.config_hash(run_cfg)}.jsonl"
    print(f"[STAGE] batch {batch_size} -> {out.name}")
    api.run_evaluation(vlm, check_items, run_cfg, R_CHECK, out, resume=True, batch_size=batch_size)
    check_rows[batch_size] = {row["item_id"]: row for row in api.read_results(out)}

In [ ]:
check_table = pd.DataFrame([
    {
        "benchmark": item.benchmark,
        "item_id": item.item_id,
        "first_answer": item.answers[0],
        "pred_b1": check_rows[1][item.item_id]["pred"],
        f"pred_b{BATCH}": check_rows[BATCH][item.item_id]["pred"],
        "raw_b1": check_rows[1][item.item_id]["raw_output"],
        f"raw_b{BATCH}": check_rows[BATCH][item.item_id]["raw_output"],
    }
    for item in check_items
])
check_table["equal"] = check_table["pred_b1"] == check_table[f"pred_b{BATCH}"]
check_table

In [ ]:
mismatch = check_table[~check_table["equal"]]
print(f"mismatches: {len(mismatch)} of {len(check_table)}")
mismatch

In [ ]:
assert mismatch.empty, f"batch {BATCH} changed {len(mismatch)} prediction(s); see the table above before the full run"
print(f"✓ batch check: batch 1 and batch {BATCH} give the same predictions on {N_CHECK} items")

### Step-2:- Full run, one shard per GPU group

`RUN_FULL` runs shard `SHARD` in this kernel, on the GPUs this kernel can see, through the same `api.run_full` the CLI calls.
Shard files carry the shard in their name, so the kernel and the CLI write the same files.

For the two-shard run, use the terminal on Inara, one shard per GPU group:

```
python scripts/mvp.py full --shard 0/2
python scripts/mvp.py full --shard 1/2
```

Each command sets `CUDA_VISIBLE_DEVICES` to its group in `gpu_groups` (GPUs 0-1 and 2-3). At most 2 model copies fit on
GPUs 0-3, and this kernel may still hold GPUs 0 and 1 from step 1, so free them first (restart the kernel, or run
`del vlm; api.unload_model()`) before a terminal run that uses group 0. Both runs are resumable: rerun the same command after an
interruption and finished items are skipped. Do not set `RUN_FULL` for a shard that a terminal is already running.

In [ ]:
if RUN_FULL:
    print(f"[STAGE] full run, shard {SHARD[0]}/{SHARD[1]}")
    api.run_full(vlm, cfg, ROOT, shard=SHARD)
else:
    print("skipped: RUN_FULL is False")

### Step-3:- Summary per benchmark x split x R

Accuracy, parse rate and seconds per item, read from every shard file of this run config (`batch_size` from the config).
dev and ctrl_train are kept apart, and each item counts once. For batched rows, s/item is batch time divided by batch size, measured on
2x A40 with balanced placement. The table is empty until result files exist.

In [ ]:
summary = api.summarize_full(cfg, ROOT)
full_table = (
    pd.DataFrame(summary)
    if summary
    else f"no result files yet for batch_size {cfg['batch_size']} in {cfg['out_dir']}"
)
full_table

### Step-4:- Cost model (stage S6)

The controller is compared at matched FLOPs, so we need the cost of every (tokens, R) choice. FLOPs are computed from the
loaded model's parameter shapes (`docs/DEC3_MVP.md` §6) and then checked against measured time. All times in this step are
measured on 2x A40 with balanced placement; the GPUs actually in use are printed below.

In [ ]:
print("[STAGE] load model (reused if step 1 loaded it)")
vlm = api.load_model(cfg)
GPUS = sorted(set(vlm.model.hf_device_map.values()))
GPU_NOTE = f"{len(GPUS)}x {' / '.join(sorted({torch.cuda.get_device_name(i) for i in GPUS}))}, {cfg['device_map']} placement"
print("GPUs in use (torch ids):", GPUS, "|", GPU_NOTE)

#### Dev items and inputs

One set of dev items serves the whole step. GQA and TextVQA alternate, so every batch mixes both. The inputs are built once,
with the same code as the evaluation runner.

In [ ]:
gqa_items = api.load_gqa(cfg, "dev", limit=N_COST // 2, root=ROOT)
textvqa_items = api.load_textvqa(cfg, "dev", limit=N_COST // 2, root=ROOT)
cost_items = [item for pair in zip(gqa_items, textvqa_items) for item in pair]
cost_inputs = [api.item_inputs(vlm, item, cfg) for item in cost_items]
prompt_lens = [inputs["input_ids"].shape[1] for inputs in cost_inputs]  # T per item, all visual tokens
assert len(cost_inputs) == N_COST, len(cost_inputs)
print(f"{N_COST} items | prompt tokens: min {min(prompt_lens)}, median {statistics.median(prompt_lens)}, max {max(prompt_lens)}")

#### Active parameters per part

Parameters used per token (multiply-adds). MoE experts count only the share that is active (8 of 128). `core` is one loop
pass; `core_message` is what every pass after the first adds; `embed` is a lookup and costs no FLOPs.

In [ ]:
params = api.cost_params(vlm.model)
PARTS = ["embed", "prelude", "core", "core_message", "coda", "lm_head", "vit_patch", "vit_layers", "vit_projector"]
pd.Series({part: asdict(params)[part] / 1e6 for part in PARTS}, name="million parameters").round(2)

#### FLOPs per R for a typical item

The typical item is the one with the median prompt length. Its output length is taken from one real answer at R = 3. The
vision tower does not depend on R or on pruning, so its share falls as R grows.

In [ ]:
typical = sorted(range(N_COST), key=lambda i: prompt_lens[i])[N_COST // 2]
typical_inputs, n_tokens = cost_inputs[typical], prompt_lens[typical]
grid_thw = typical_inputs["image_grid_thw"]
api.set_loops(vlm, api.REF_R)
n_out = len(api.generate(vlm, typical_inputs, cfg["max_new_tokens"]).token_ids)
print(f"item {cost_items[typical].item_id} | prompt tokens {n_tokens} | output tokens {n_out} | image grid {grid_thw.tolist()}")

In [ ]:
flops_by_R = pd.DataFrame(
    [api.flops_total(params, n_tokens, n_out, R, grid_thw) for R in cfg["R_grid"]],
    index=pd.Index(cfg["R_grid"], name="R"),
)
flops_table = (flops_by_R / 1e9).add_suffix(" (G)")
flops_table["vit_share"] = flops_by_R["flops_vit"] / flops_by_R["flops_total"]
flops_table.round(3)

In [ ]:
assert flops_by_R["flops_total"].is_monotonic_increasing and flops_by_R["flops_vit"].nunique() == 1
assert flops_by_R.loc[api.REF_R, "flops_total"] == api.cost_ref(params, n_tokens, n_out, grid_thw)
print("✓ cost model: FLOPs grow with R, ViT cost is fixed, R = 3 equals cost_ref")

#### Measured vs analytic prefill time ratio

For each item the prompt forward is timed (time to first token) at both loop counts. All visual tokens are kept through
`prune_inputs`, so the vision tower runs outside the timer. One warmup call, then the median of the repeats. Keep the GPUs
otherwise idle. The test of stage S6: the median measured ratio is within ±25% of the analytic one.

In [ ]:
R_LOW, R_HIGH = R_RATIO
print(f"[STAGE] prefill timing at R={R_LOW} and R={R_HIGH} on {N_COST} items | measured on {GPU_NOTE}")
ratio_rows = []
for i, (item, inputs, n) in enumerate(zip(cost_items, cost_inputs, prompt_lens)):
    low_s = api.time_prefill(vlm, inputs, R_LOW, N_REPEATS)
    high_s = api.time_prefill(vlm, inputs, R_HIGH, N_REPEATS)
    analytic = api.flops_prefill(params, n, R_HIGH) / api.flops_prefill(params, n, R_LOW)
    ratio_rows.append({"item_id": item.item_id, "n_tokens": n, "low_s": low_s, "high_s": high_s,
                       "measured": high_s / low_s, "analytic": analytic})
    print(f"[{i + 1}/{N_COST}] T={n} | R={R_LOW}: {low_s:.3f}s | R={R_HIGH}: {high_s:.3f}s | ratio {high_s / low_s:.2f}")
api.set_loops(vlm, api.REF_R)
ratios = pd.DataFrame(ratio_rows)
ratios

In [ ]:
measured, analytic = ratios["measured"].median(), ratios["analytic"].median()
print(f"median R{R_HIGH}/R{R_LOW} prefill ratio | measured {measured:.2f} | analytic {analytic:.2f} | measured on {GPU_NOTE}")
assert abs(measured / analytic - 1) <= RATIO_TOL, f"measured {measured:.2f} vs analytic {analytic:.2f}"
print(f"✓ test_flops: measured prefill time ratio is within ±{RATIO_TOL:.0%} of the analytic ratio")

#### Throughput: batch 1 vs batch 8

Seconds per item for the same preloaded inputs at one loop count, with `generate` (batch 1) and `generate_batch` (the
config's `batch_size`, 8). Wall clock, including the vision tower and decoding, excluding data loading. One warmup call per
batch size. A batch runs until its longest answer ends. Measured on 2x A40 with balanced placement; these seconds set the
item counts of the grid.

In [ ]:
print(f"[STAGE] throughput at R={R_THROUGHPUT}, batch sizes {THROUGHPUT_BATCHES} | measured on {GPU_NOTE}")
throughput = pd.DataFrame([
    api.time_throughput(vlm, cost_inputs, R_THROUGHPUT, batch_size, cfg["max_new_tokens"])
    for batch_size in THROUGHPUT_BATCHES
])
throughput["gpus"] = GPU_NOTE
throughput["config_hash"] = api.config_hash(cfg)
throughput["git_hash"] = api.git_hash()
api.set_loops(vlm, api.REF_R)
throughput

In [ ]:
throughput_path = ROOT / cfg["out_dir"] / f"throughput_R{R_THROUGHPUT}_n{N_COST}_{api.config_hash(cfg)}.csv"
throughput_path.parent.mkdir(parents=True, exist_ok=True)
throughput.to_csv(throughput_path, index=False)
assert throughput_path.is_file() and (throughput["s_per_item"] > 0).all()
for row in throughput.itertuples():
    print(f"batch {row.batch_size}: {row.s_per_item:.2f} s/item | measured on {row.gpus}")
print(f"✓ throughput saved to {throughput_path.relative_to(ROOT)}")

### Step-5:- Figure F04

Cost model: FLOPs per loop count R for one typical item (median prompt length), and the analytic against measured prefill time ratio from step 4. The heatmap is one row (all visual tokens kept).

In [ ]:
api.set_paper_style()
fig = api.plot_cost_model(flops_by_R.reset_index().assign(k=1.0), ratios, R_RATIO, RATIO_TOL)

In [ ]:
pdf_path, png_path = api.save_figure(fig, "F04_cost_model", final=False)

In [ ]:
assert pdf_path.is_file() and png_path.is_file(), f"missing {pdf_path} or {png_path}"
print("✓ F04_cost_model draft saved as PDF and PNG (final=False, paper/ untouched)")

### Step-6:- Figure F05

Accuracy against loop passes R with all visual tokens, one panel per benchmark (the loop-depth sweep). Check `n` per row before using the figure: a partial run looks like a full one, so every row must have the full item count.

In [ ]:
pd.DataFrame(summary)[["benchmark", "split", "R", "n", "accuracy"]] if summary else "skipped: no result files yet (run step 2 first)"

In [ ]:
if summary:
    api.set_paper_style()
    fig = api.plot_accuracy_vs_r(pd.DataFrame(summary))
else:
    print("skipped: summary is empty, no result files yet")

In [ ]:
if summary:
    pdf_path, png_path = api.save_figure(fig, "F05_accuracy_vs_r", final=False)
else:
    print("skipped: F05 not saved, summary is empty")

In [ ]:
if summary:
    assert pdf_path.is_file() and png_path.is_file(), f"missing {pdf_path} or {png_path}"
    print("✓ F05_accuracy_vs_r draft saved as PDF and PNG (final=False, paper/ untouched)")
else:
    print("skipped: F05 not saved, summary is empty")